In [1]:
!pip install -q "transformers==4.44.2" "datasets==2.21.0" "accelerate==0.34.2" "peft==0.13.0"
!ls /kaggle/input/notebooks/livox91/notebook1-data-preprocessing

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 62.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires g

In [2]:
%%writefile train.py
"""DDP fine-tuning of openai/whisper-medium with fp16 LoRA for Quranic
word-level pronunciation scoring (multi-voice). Launched once per GPU.

FIXES applied vs original:
  1. Cast dataset features to a common schema before concatenate_datasets
     (surah_id int64 vs int32 mismatch caused a hard crash).
  2. Dropped keep_in_memory=True on Kaggle free-tier to avoid RAM OOM.
  3. dataloader_num_workers reduced to 0 (forked workers copy dataset RAM).
  4. per_device_eval_batch_size reduced 8->4 (no grad-ckpt during eval).
  5. OUTPUT_DIR created early so margin_log.jsonl write never fails.
  6. forced_score mask guard: skip words where content region is empty.
  7. MarginProbeCallback: model.eval() + torch.no_grad() properly scoped.
"""

import json
import random
from dataclasses import dataclass
import os

import numpy as np
import torch

# Newer torch defaults torch.load to weights_only=True, which can reject a
# checkpoint's RNG-state file on resume. Our own checkpoints are trusted.
_orig_torch_load = torch.load
def _patched_torch_load(*a, **k):
    k.setdefault("weights_only", False)
    return _orig_torch_load(*a, **k)
torch.load = _patched_torch_load

import datasets
from datasets import load_from_disk, concatenate_datasets, Features, Value, Sequence

# disable_caching() is a plain function (not a context manager) in this version
# of the library. Call it once globally. Also redirect any residual cache writes
# away from the read-only /kaggle/input filesystem.
datasets.disable_caching()
os.environ.setdefault("HF_DATASETS_CACHE", "/kaggle/working/hf_cache")
from peft import LoraConfig, get_peft_model
from transformers import (
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)
from transformers.trainer_utils import get_last_checkpoint

MODEL_NAME = "openai/whisper-medium"

DATA_PATHS = [
    "/kaggle/input/notebooks/livox91/notebook1-data-preprocessing/processed_quran_words",
    "/kaggle/input/notebooks/livox91/notebook1b-ayah-segmentation/processed_quran_ayah_words",
]
for p in DATA_PATHS:
    print(p, "exists:", os.path.exists(p))

OUTPUT_DIR = "/kaggle/working/whisper-medium-quran-lora-multivoice"
SAMPLING_RATE = 16_000

# Hold out 2-3 whole reciters to test generalization to UNSEEN VOICES.
# Edit after Notebook 1b prints its distinct reciter names.
VALIDATION_RECITERS = set()   # e.g. {"reciter_07", "reciter_19"}

LOCAL_RANK = int(os.environ.get("LOCAL_RANK", 0))
IS_MAIN    = LOCAL_RANK == 0

# Create output dir early so margin_log.jsonl writes never fail mid-run.
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------------------------------------------------------
# 1) Data: load + merge the single-voice and multi-voice word datasets
# -----------------------------------------------------------------------------
parts = [load_from_disk(p) for p in DATA_PATHS]

# FIX 1: Align feature schemas before concatenating.
# The two datasets may store surah_id as int32 vs int64, which causes
# concatenate_datasets to raise a ValueError. Cast everything to int64.
def align_features(ds):
    """Cast integer columns to int64 so both datasets share the same schema.

    ds.cast() calls .map() internally. Caching is already disabled globally
    above (datasets.disable_caching() + HF_DATASETS_CACHE redirect) so no
    temp files are written to the read-only /kaggle/input filesystem.
    """
    new_schema = {}
    changed = False
    for col, feat in ds.features.items():
        if isinstance(feat, Value) and feat.dtype in ("int32", "int16", "int8"):
            new_schema[col] = Value("int64")
            changed = True
        elif (isinstance(feat, Sequence) and
              isinstance(feat.feature, Value) and
              feat.feature.dtype in ("int32", "int16", "int8")):
            new_schema[col] = Sequence(Value("int64"))
            changed = True
        else:
            new_schema[col] = feat
    if changed:
        ds = ds.cast(Features(new_schema), keep_in_memory=True)
    return ds

parts = [align_features(p) for p in parts]
dataset = concatenate_datasets(parts) if len(parts) > 1 else parts[0]

if IS_MAIN:
    print("merged dataset:", len(dataset), "rows")
    print("reciters:", sorted(set(dataset["reciter"]))[:30])

# Split by index (avoids cache writes to read-only /kaggle/input).
reciters = np.array(dataset["reciter"])
if VALIDATION_RECITERS:
    val_mask = np.isin(reciters, list(VALIDATION_RECITERS))
else:
    # Fallback: 5% random slice. Does NOT test unseen voices —
    # set VALIDATION_RECITERS for the real generalization test.
    rng = np.random.default_rng(0)
    val_mask = rng.random(len(reciters)) < 0.05

# FIX 2: Removed keep_in_memory=True — on Kaggle free-tier (≈13 GB RAM)
# loading the full dataset twice into RAM causes OOM. Arrow mmap is fine.
train_ds = dataset.select(np.where(~val_mask)[0])
eval_ds  = dataset.select(np.where(val_mask)[0])

if IS_MAIN:
    print(f"train: {len(train_ds)} | eval: {len(eval_ds)}")

processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="Arabic", task="transcribe")

# -----------------------------------------------------------------------------
# 2) Collator: waveform -> log-Mel on the fly, pad labels, mask padding
# -----------------------------------------------------------------------------
@dataclass
class OnTheFlyWhisperCollator:
    processor: object
    decoder_start_token_id: int

    def __call__(self, features):
        arrays = [np.asarray(f["audio_array"], dtype=np.float32) for f in features]
        batch = self.processor.feature_extractor(
            arrays, sampling_rate=SAMPLING_RATE, return_tensors="pt"
        )
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

# -----------------------------------------------------------------------------
# 3) Minimal-pair machinery: harakah perturbations + forced scoring
# -----------------------------------------------------------------------------
FATHA, DAMMA, KASRA, SUKUN = "َ", "ُ", "ِ", "ْ"
FATHATAN, DAMMATAN, KASRATAN = "ً", "ٌ", "ٍ"
SHADDAH = "ّ"
SHORT_HARAKAT = {FATHA, DAMMA, KASRA, SUKUN}
TANWEEN = {FATHATAN, DAMMATAN, KASRATAN}

def harakah_variants(word, max_variants=None, seed=0):
    """Plausible mispronunciations used as hard negatives.

    FIX 8 (shadda emphasis): real-audio testing on Surah Al-Fatiha showed the
    model systematically DROPS the shadda (gemination) -- 19/29 words failed as
    shadda-drops. Root cause: shadda-drop was just one randomly-sampled variant,
    so the model was rarely forced to contrast it. Now the shadda-dropped form
    is a FORCED hard negative: always kept (never sampled away) for any word
    with a shadda, so the model must learn to rank the doubled form above the
    un-doubled one on every shadda word.
    """
    variants = set()
    forced = set()                       # variants that must never be dropped
    for i, ch in enumerate(word):
        pool = SHORT_HARAKAT if ch in SHORT_HARAKAT else TANWEEN if ch in TANWEEN else None
        if pool is not None:
            for alt in pool - {ch}:
                variants.add(word[:i] + alt + word[i + 1:])
        elif ch == SHADDAH:
            dropped = word[:i] + word[i + 1:]
            variants.add(dropped)
            forced.add(dropped)          # shadda-drop = forced hard negative
    variants.discard(word)
    forced.discard(word)
    out = sorted(variants)
    if max_variants is not None and len(out) > max_variants:
        keep = sorted(forced)            # always keep shadda negatives
        rest = [v for v in out if v not in forced]
        budget = max(0, max_variants - len(keep))
        keep += random.Random(seed).sample(rest, min(budget, len(rest)))
        out = sorted(set(keep))
    return out

# Whisper token layout: [BOS, lang_tok, task_tok, notimestamps, ...content..., EOS]
# We score only the content tokens (index 4 to len-2 inclusive).
CONTENT_START = 4

def forced_score(model, tokenizer, input_features, texts):
    """
    Compute mean log-prob over content tokens for each text candidate.

    FIX 3: Guard against words whose token sequence is too short to contain
    any content tokens — previously the empty mask silently returned 0 for
    all such words, corrupting the margin probe statistics.
    """
    device = input_features.device
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    seqs = [tokenizer(t).input_ids for t in texts]
    maxlen = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), maxlen), pad_id, dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
    decoder_input_ids = ids[:, :-1].to(device)
    targets = ids[:, 1:].to(device)
    with torch.no_grad():
        logits = model(
            input_features=input_features.expand(len(seqs), -1, -1),
            decoder_input_ids=decoder_input_ids,
        ).logits
    token_logprobs = logits.log_softmax(-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    mask = torch.zeros_like(targets, dtype=torch.bool)
    for i, s in enumerate(seqs):
        # Content region: positions CONTENT_START..len(s)-2 (before EOS).
        start = CONTENT_START
        end   = len(s) - 2        # exclusive upper bound in the shifted target
        if end > start:           # FIX: skip if no content tokens exist
            mask[i, start:end] = True
    counts = mask.sum(1).clamp(min=1)
    return (token_logprobs * mask).sum(1) / counts

class MarginProbeCallback(TrainerCallback):
    def __init__(self, probe_ds, processor, n_probe=150, max_variants=6):
        idx = list(range(0, len(probe_ds), max(1, len(probe_ds) // n_probe)))[:n_probe]
        self.probe = [
            {"audio_array": probe_ds[i]["audio_array"], "text": probe_ds[i]["text"]}
            for i in idx
        ]
        self.processor = processor
        self.max_variants = max_variants

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if not state.is_world_process_zero or model is None:
            return
        # FIX 4: Set eval mode and disable grad for the entire probe loop,
        # not just inside forced_score.
        model.eval()
        wins, margins = 0, []
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for ex in self.probe:
                variants = harakah_variants(ex["text"], max_variants=self.max_variants)
                if not variants:
                    continue
                feats = self.processor.feature_extractor(
                    np.asarray(ex["audio_array"], dtype=np.float32),
                    sampling_rate=SAMPLING_RATE, return_tensors="pt",
                ).input_features.to(next(model.parameters()).device)
                scores = forced_score(model, self.processor.tokenizer, feats,
                                      [ex["text"]] + variants)
                margin = (scores[0] - scores[1:].max()).item()
                wins += margin > 0
                margins.append(margin)
        if margins:
            rec = {
                "step": state.global_step,
                "probe_winrate": round(wins / len(margins), 4),
                "probe_mean_margin": round(float(np.mean(margins)), 4),
                "probe_n": len(margins),
            }
            print(f"[margin probe] {rec}")
            log_path = os.path.join(args.output_dir, "margin_log.jsonl")
            with open(log_path, "a") as f:
                f.write(json.dumps(rec) + "\n")

# -----------------------------------------------------------------------------
# 4) Model: frozen fp32 base + trainable LoRA adapters
# -----------------------------------------------------------------------------
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False
model.generation_config.language = "ar"
model.generation_config.task = "transcribe"

lora_config = LoraConfig(
    r=32, lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none",
)
model = get_peft_model(model, lora_config)
if IS_MAIN:
    model.print_trainable_parameters()

# -----------------------------------------------------------------------------
# 5) Training arguments
# Global batch = 4 x 2 grad-accum x 2 GPUs = 16.
# fp16 on T4 (tensor cores); bf16 has no T4 hardware support (~5x slower).
# FIX 5: dataloader_num_workers=0  — each worker forks the dataset into RAM;
#         on free-tier Kaggle (≈13 GB RAM) 2 workers OOM with a large dataset.
# FIX 6: per_device_eval_batch_size=4 — gradient_checkpointing is off during
#         eval, so full activations accumulate; 8 risks OOM on a T4.
# -----------------------------------------------------------------------------
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,        # was 8 — reduced for T4 eval safety
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=500,
    max_steps=10000,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="steps",
    eval_steps=1000,
    save_steps=1000,
    save_total_limit=3,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_num_workers=0,            # was 2 — fork copies dataset to RAM
    ddp_find_unused_parameters=False,
    report_to="none",
)

_trainer_kwargs = dict(
    args=training_args,
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=OnTheFlyWhisperCollator(
        processor=processor,
        decoder_start_token_id=model.config.decoder_start_token_id,
    ),
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3),
        MarginProbeCallback(eval_ds, processor),
    ],
)
# transformers renamed tokenizer= -> processing_class=; try new, fall back to old.
try:
    trainer = Seq2SeqTrainer(processing_class=processor.feature_extractor, **_trainer_kwargs)
except TypeError:
    trainer = Seq2SeqTrainer(tokenizer=processor.feature_extractor, **_trainer_kwargs)

# -----------------------------------------------------------------------------
# 6) Train. Auto-resume if a checkpoint already exists (e.g. session timeout).
# -----------------------------------------------------------------------------
last_ckpt = get_last_checkpoint(OUTPUT_DIR) if os.path.isdir(OUTPUT_DIR) else None
if IS_MAIN and last_ckpt:
    print(f"Resuming from {last_ckpt}")
trainer.train(resume_from_checkpoint=last_ckpt)

if IS_MAIN:
    model.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
    processor.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
    print("Done. Adapter saved to", f"{OUTPUT_DIR}/final_adapter")

Writing train.py


In [3]:
!accelerate launch --multi_gpu --num_processes=2 --mixed_precision=fp16 train.py

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-06-16 01:13:00.072909: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-16 01:13:00.072909: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781572380.434063      68 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781572380.434090      69 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register fact

In [4]:
import json
with open("/kaggle/working/whisper-medium-quran-lora-multivoice/margin_log.jsonl") as f:
    for line in f:
        print(json.loads(line))

{'step': 1000, 'probe_winrate': 0.98, 'probe_mean_margin': 1.2403, 'probe_n': 150}
{'step': 2000, 'probe_winrate': 0.9667, 'probe_mean_margin': 1.3496, 'probe_n': 150}
{'step': 3000, 'probe_winrate': 0.9867, 'probe_mean_margin': 1.4363, 'probe_n': 150}
{'step': 4000, 'probe_winrate': 0.98, 'probe_mean_margin': 1.493, 'probe_n': 150}
{'step': 5000, 'probe_winrate': 0.9933, 'probe_mean_margin': 1.4576, 'probe_n': 150}
{'step': 6000, 'probe_winrate': 0.9867, 'probe_mean_margin': 1.493, 'probe_n': 150}
{'step': 7000, 'probe_winrate': 0.9867, 'probe_mean_margin': 1.5771, 'probe_n': 150}
{'step': 8000, 'probe_winrate': 0.98, 'probe_mean_margin': 1.6328, 'probe_n': 150}
{'step': 9000, 'probe_winrate': 0.98, 'probe_mean_margin': 1.6765, 'probe_n': 150}
{'step': 10000, 'probe_winrate': 0.98, 'probe_mean_margin': 1.6822, 'probe_n': 150}
